# SymPy: working with mathematics symbolically

A rectangle must use a fixed amount of fencing while maximizing area. Derive the relationship exactly, define the feasible domain and verify the candidate optimum rather than relying on a plot alone. Earlier examples develop the algebra, calculus and checking habits used in that derivation.

A numerical calculation gives a value. A symbolic calculation keeps variables and exact relationships, letting us factor an expression, differentiate a function, or solve an equation before choosing numerical inputs.

We will start with exact arithmetic, build algebra and calculus examples, and finish with a rectangle optimization problem that connects a symbolic solution to a numerical plot. Every example runs offline after installation. Run cells from top to bottom.

## 1. Install and create symbols

If needed, run `%pip install sympy numpy matplotlib` in a separate cell. A Symbol represents an unknown mathematical quantity. Python variable names refer to those symbolic objects; assigning a Python value later does not rewrite existing expressions.

Use `**` for powers. Python's `^` operator means something different.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

print("SymPy:", sp.__version__)
x, y = sp.symbols("x y", real=True)
expression = x**2 + 2*x + 1
print("Expression:", expression)
print("Symbols:", expression.free_symbols)

## 2. Keep exact numbers exact

Python evaluates `1 / 3` as a floating-point number before SymPy sees it. `Rational(1, 3)` preserves the exact fraction. Constants such as `pi` also remain symbolic until numerical evaluation is requested.

`evalf(30)` requests 30 significant decimal digits; it does not create missing precision in an ordinary Python float. Use exact fractions or decimal strings when constructing high-precision values.

In [ ]:
exact = sp.Rational(1, 3) + sp.Rational(1, 6)
print("Exact sum:", exact)
print("Python division:", 1 / 3)
print("Pi to 30 digits:", sp.pi.evalf(30))
print("Exact decimal fraction:", sp.Rational("0.1"))
print("Symbolic square root:", sp.sqrt(2))

## 3. Change an expression’s form without forgetting its domain

Predict which form makes roots or coefficients easiest to inspect. After simplification, check values excluded by the original denominator. Equivalent-looking formulas may still need the original domain restriction.

Different forms answer different questions. Expanded polynomials make coefficients visible; factored forms make roots easier to see. `simplify` tries several transformations, while a targeted operation such as `factor` states what form you want.

Simplification can hide an expression's original domain restrictions. Cancelling $(x^2-1)/(x-1)$ gives $x+1$, but the original quotient is still undefined at $x=1$.

In [ ]:
expanded = sp.expand((x + 1)**2)
factored = sp.factor(x**2 - 1)
trig_identity = sp.simplify(sp.sin(x)**2 + sp.cos(x)**2)
quotient = (x**2 - 1) / (x - 1)
print("Expanded:", expanded)
print("Factored:", factored)
print("Trigonometric identity:", trig_identity)
print("Cancelled quotient:", sp.cancel(quotient), "with original restriction x != 1")

## 4. Substitute without changing the original

`subs` returns a new expression. It can replace a symbol with a number or another expression. Multiple replacements can depend on order; `simultaneous=True` is useful when swapping symbols.

Check the unchanged original to distinguish symbolic substitution from mutation.

In [ ]:
at_three = expression.subs(x, 3)
shifted = expression.subs(x, y + 1)
swapped = (x - y).subs({x: y, y: x}, simultaneous=True)
print("At x=3:", at_three)
print("Replace x with y+1:", sp.expand(shifted))
print("Swap x and y:", swapped)
print("Original:", expression)

## 5. Assumptions affect valid simplifications

For real $x$, $\sqrt{x^2}=|x|$, not always $x$. For a positive symbol, the absolute value can be removed. Unspecified symbols may be complex, so SymPy avoids transformations that would be invalid in that domain.

Create assumptions with the symbol. Symbols with the same printed name but different assumptions are distinct objects.

In [ ]:
positive_length = sp.Symbol("length", positive=True)
complex_z = sp.Symbol("z")
print("Real x:", sp.sqrt(x**2))
print("Positive length:", sp.sqrt(positive_length**2))
print("Unrestricted z:", sp.sqrt(complex_z**2))

## 6. Express and solve equations

Use `Eq(left, right)` to represent an equation. `=` assigns a Python variable; `==` generally compares symbolic structure, not mathematical equivalence.

`solveset` returns a set and uses an explicit domain. An expression without `Eq` is treated as equal to zero. Some problems return an unevaluated condition set when the solver cannot express the solution explicitly; that is not the same as an empty solution set.

In [ ]:
equation = sp.Eq(x**2 - 5*x + 6, 0)
roots = sp.solveset(equation, x, domain=sp.S.Reals)
print("Equation:", equation)
print("Real roots:", roots)
print("x²+1=0 over reals:", sp.solveset(x**2 + 1, x, domain=sp.S.Reals))
print("x²+1=0 over complexes:", sp.solveset(x**2 + 1, x, domain=sp.S.Complexes))
print("Structural equality:", (x + 1)**2 == x**2 + 2*x + 1)
print("Simplified difference:", sp.simplify((x + 1)**2 - (x**2 + 2*x + 1)))

## 7. Substitute a candidate into every original equation

Before accepting the returned tuple, match its positions to the variable order. Substitute it into both sides of each equation. A solver output is more useful when its residual can be checked independently.

`linsolve` handles linear systems. The variable sequence specifies the order of values in each solution tuple. A solution should satisfy every original equation, so substitution provides a direct check.

In [ ]:
system = [sp.Eq(2*x + y, 7), sp.Eq(x - y, 2)]
linear_solutions = sp.linsolve(system, (x, y))
solution_tuple = next(iter(linear_solutions))
solution_map = dict(zip((x, y), solution_tuple))
print("Solutions (x, y):", linear_solutions)
print("Equation checks:", [eq.subs(solution_map) for eq in system])

## 8. Differentiate a function

A derivative describes local change. Repeated differentiation gives higher derivatives; specifying the variable matters when an expression contains several symbols.

For a multivariable function, a partial derivative treats the other independent symbols as constant.

In [ ]:
function = x**3 - 3*x**2 + 2
first_derivative = sp.diff(function, x)
second_derivative = sp.diff(function, x, 2)
print("Function:", function)
print("First derivative:", first_derivative)
print("Second derivative:", second_derivative)
print("Partial derivative with respect to x:", sp.diff(x**2 * y + sp.sin(y), x))

## 9. Integrate and check by differentiation

An indefinite integral returns an antiderivative without an arbitrary additive constant. A definite integral evaluates signed accumulation between bounds; it is not always geometric area because negative values subtract.

Differentiating an antiderivative should recover the integrand where the expressions are valid. Some integrals remain unevaluated when SymPy cannot find an explicit form.

In [ ]:
integrand = 3*x**2 + 2*x
antiderivative = sp.integrate(integrand, x)
definite_integral = sp.integrate(integrand, (x, 0, 2))
print("Antiderivative, plus a constant:", antiderivative)
print("Integral from 0 to 2:", definite_integral)
print("Derivative check:", sp.simplify(sp.diff(antiderivative, x) - integrand))

## 10. Limits and local series

A limit can exist even when direct substitution is undefined. A series describes local behavior near an expansion point. `O(x**6)` records the omitted order; removing it produces a polynomial approximation, not an exact identity.

Approximation quality depends on where it is evaluated. Compare a point near zero with one farther away.

In [ ]:
sinc_limit = sp.limit(sp.sin(x) / x, x, 0)
sine_series = sp.series(sp.sin(x), x, 0, 6)
sine_polynomial = sine_series.removeO()
print("Limit sin(x)/x at zero:", sinc_limit)
print("Sine series:", sine_series)
for value in [sp.Rational(1, 10), sp.Integer(2)]:
    error = sp.Abs(sp.sin(value) - sine_polynomial.subs(x, value)).evalf()
    print(f"Approximation error at {value}: {error}")

## 11. Check which numerical root your starting guess found

Inspect the function near the initial guess and predict a plausible root region. After solving, examine the residual. A small residual supports this root but does not prove that no other root exists.

`nsolve` finds a numerical root near a supplied starting guess. It does not enumerate every root or prove uniqueness. Check the residual after solving, and inspect the domain before accepting a result.

For $\cos(x)=x$, we start at 0.7 and report the residual.

In [ ]:
numerical_root = sp.nsolve(sp.cos(x) - x, sp.Float("0.7"), prec=30)
root_residual = (sp.cos(x) - x).subs(x, numerical_root).evalf()
print("Numerical root:", numerical_root)
print("Residual:", root_residual)

## 12. Work with exact matrices

A SymPy Matrix supports exact symbolic linear algebra. Solve $A v=b$ directly with `LUsolve`; computing an inverse is unnecessary for this task.

This matrix is nonsingular, so the solution is unique. Singular systems need a method that can describe inconsistency or free parameters.

In [ ]:
A = sp.Matrix([[2, 1], [1, -1]])
b = sp.Matrix([7, 2])
vector_solution = A.LUsolve(b)
print("Determinant:", A.det())
print("Solution vector:", vector_solution)
print("Residual vector:", A * vector_solution - b)
print("Eigenvalues with multiplicities:", A.eigenvals())

## 13. Solve a differential equation

An undefined symbolic function such as `u(t)` can be differentiated and used in an equation. This example describes exponential decay: $u'(t)=-2u(t)$ with $u(0)=3$.

`dsolve` returns an equation. Substitute its right-hand side into the differential equation and initial condition to verify the solution.

In [ ]:
t = sp.Symbol("t", real=True)
u = sp.Function("u")
ode = sp.Eq(sp.diff(u(t), t), -2*u(t))
ode_solution = sp.dsolve(ode, ics={u(0): 3})
print("Solution:", ode_solution)
print("Differential-equation residual:", sp.simplify(sp.diff(ode_solution.rhs, t) + 2*ode_solution.rhs))
print("Initial value:", ode_solution.rhs.subs(t, 0))

## 14. Bridge symbolic work and NumPy

`lambdify` creates a numerical function from a symbolic expression. Use SymPy functions while deriving expressions and NumPy functions or the lambdified function when evaluating arrays.

This lets us differentiate exactly, then evaluate many points efficiently for a chart. The numerical arrays contain floating-point approximations.

In [ ]:
numeric_function = sp.lambdify(x, function, modules="numpy")
numeric_derivative = sp.lambdify(x, first_derivative, modules="numpy")
x_values = np.linspace(-1, 3, 200)
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True, layout="constrained")
axes[0].plot(x_values, numeric_function(x_values))
axes[0].set(title="A polynomial and its exact derivative", ylabel="f(x)")
axes[1].plot(x_values, numeric_derivative(x_values), color="tab:orange")
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].set(xlabel="x", ylabel="f′(x)")
plt.show()

## 15. Compare an interior optimum with the feasible boundaries

Write the allowed width interval before differentiating. Predict why zero width cannot maximize a positive area. Compare stationary points and boundary values, then use curvature to explain the optimum rather than merely naming it.

Suppose a rectangle has 40 meters of perimeter. With width $w$ and length $l$, $2w+2l=40$, so $l=20-w$ and $A(w)=w(20-w)$.

Feasible nondegenerate rectangles have $0<w<20$. A stationary point alone is not enough: inspect curvature and compare with the boundary areas on the closed interval. Both boundaries have zero area and represent degenerate rectangles. A negative second derivative throughout the interval makes the interior stationary point the unique maximum.

In [ ]:
w = sp.Symbol("w", real=True)
length_expression = 20 - w
area = w * length_expression
area_slope = sp.diff(area, w)
stationary_points = sp.solveset(area_slope, w, domain=sp.Interval(0, 20))
optimal_width = next(iter(stationary_points))
optimal_length = length_expression.subs(w, optimal_width)
maximum_area = area.subs(w, optimal_width)
print("Area expression:", sp.expand(area))
print("Stationary points:", stationary_points)
print("Second derivative:", sp.diff(area, w, 2))
print("Boundary areas:", area.subs(w, 0), area.subs(w, 20))
print(f"Optimal width: {optimal_width} m; length: {optimal_length} m; area: {maximum_area} m²")

## 16. Plot and export the result

Evaluate the exact area expression across feasible widths, then mark the symbolic optimum. Export the plot as PNG and SVG, and save both a readable solution and LaTeX markup for the formula.

This cell creates `outputs` and replaces these named files if they already exist. The LaTeX file contains math markup for insertion into a document, not a standalone compiled document.

In [ ]:
area_function = sp.lambdify(w, area, modules="numpy")
widths = np.linspace(0, 20, 201)
report, ax = plt.subplots(figsize=(8, 5), layout="constrained")
ax.plot(widths, area_function(widths), label="Area with 40 m perimeter")
ax.scatter([float(optimal_width)], [float(maximum_area)], color="tab:red", zorder=3, label="Maximum")
ax.annotate(f"{optimal_width} m × {optimal_length} m = {maximum_area} m²",
            xy=(float(optimal_width), float(maximum_area)), xytext=(11, 75),
            arrowprops={"arrowstyle": "->"})
ax.set(xlabel="Width (m)", ylabel="Area (m²)", title="Rectangle area under a fixed perimeter", xlim=(0, 20), ylim=(0, 115))
ax.legend(loc="lower center")
output_dir = Path.cwd() / "outputs"
output_dir.mkdir(exist_ok=True)
for extension in ["png", "svg"]:
    destination = output_dir / f"sympy_rectangle_report.{extension}"
    report.savefig(destination, dpi=150, bbox_inches="tight")
    print("Saved:", destination)
solution_text = (f"Perimeter: 40 m\nArea(w): {area}\nDerivative: {area_slope}\n"
                 f"Optimal width: {optimal_width} m\nOptimal length: {optimal_length} m\nMaximum area: {maximum_area} m²\n")
(output_dir / "sympy_rectangle_solution.txt").write_text(solution_text, encoding="utf-8")
(output_dir / "sympy_rectangle_formula.tex").write_text(sp.latex(sp.Eq(sp.Symbol("A"), area)), encoding="utf-8")
print("Saved solution text and LaTeX formula.")
plt.show()

## 17. Practice and common mistakes

1. Replace 40 with a positive symbolic perimeter $P$. Derive the optimal dimensions and area in terms of $P$.
2. Compare the sine series with the sine function over increasingly wide intervals.
3. Solve a linear system with infinitely many solutions, then one with no solutions. Inspect the returned sets.
4. Change the differential equation to $u'=-ku$ with positive $k$ and verify the general decay form.

| Observation | What to check |
| --- | --- |
| Unexpected decimals | Python may have evaluated arithmetic before SymPy; use exact integers and `Rational` |
| Algebraically equal expressions compare unequal | Structural equality differs from simplifying their difference |
| Square roots do not simplify as expected | Check assumptions and domains |
| An apparent root is invalid | Substitute into the original equation and check denominator restrictions |
| `nsolve` finds only one root | It is a local numerical solver; starting values and domain matter |
| A transformation does not change the expression | Assign the returned expression; SymPy expressions are immutable |
| NumPy fails on symbolic expressions | Use `lambdify` to bridge symbolic and numerical work |
| A solver returns an unevaluated object | The algorithm may not find an explicit answer; this does not establish nonexistence |

## Further reading

See SymPy's official [basic operations](https://docs.sympy.org/latest/tutorials/intro-tutorial/basic_operations.html), [solvers](https://docs.sympy.org/latest/tutorials/intro-tutorial/solvers.html), [calculus](https://docs.sympy.org/latest/tutorials/intro-tutorial/calculus.html), and [lambdify](https://docs.sympy.org/latest/modules/utilities/lambdify.html) documentation.

## Explain what the evidence supports

Express the best rectangle dimensions in terms of the perimeter. Verify the constraint and explain how the derivative, curvature and boundary comparison jointly support the result.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: derive a packaging design and evaluate many messy orders

A packaging team needs a square-base closed box for each required volume. Top/bottom material costs twice as much per square centimeter as side material. Derive the continuous minimum-material design, quarantine invalid volume strings, and compare practical rounded dimensions with cube-shaped boxes. This is an explicitly simplified engineering model: no seams, strength, cushioning or manufacturing cost is modeled.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through the constraint before differentiating

Let base width be w, height h and volume V. The constraint is w²h=V, so h=V/w² with w>0. If base material has relative cost b and side material s, total cost is 2bw²+4sV/w. Its derivative is 4bw−4sV/w², giving w³=sV/b. The cost diverges at both positive-domain ends, so the positive stationary point is a global minimum in this model, not just an arbitrary derivative zero.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

import sympy as sp
p2_w,p2_V,p2_b,p2_s=sp.symbols('w V b s',positive=True)
p2_cost=2*p2_b*p2_w**2+4*p2_s*p2_V/p2_w
p2_optimal=(p2_s*p2_V/p2_b)**sp.Rational(1,3)
assert sp.simplify(sp.diff(p2_cost,p2_w).subs(p2_w,p2_optimal))==0
assert sp.simplify(sp.diff(p2_cost,p2_w,2)-(4*p2_b+8*p2_V*p2_s/p2_w**3))==0
p2_width_fn=sp.lambdify((p2_V,p2_b,p2_s),p2_optimal,'numpy')
p2_raw_volumes=['1000','8000','bad','0','-25','3375','27000']
p2_volumes=[];p2_rejections=[]
for p2_i,p2_value in enumerate(p2_raw_volumes):
    try:
        p2_parsed=float(p2_value)
        if not np.isfinite(p2_parsed) or p2_parsed<=0:raise ValueError('volume must be finite and positive')
        p2_volumes.append(p2_parsed)
    except ValueError as p2_error:p2_rejections.append({'row':p2_i,'value':p2_value,'reason':str(p2_error)})
assert len(p2_volumes)==4 and len(p2_rejections)==3

### Work through manufacturing increments and computation cost

The continuous optimum is a lower bound. Round width upward to 0.5 cm, then recompute and round height upward so the requested volume still fits. Rounding width and height independently from unrelated designs can violate capacity. Compare with an upward-rounded cube using the same material costs.

Derive once symbolically, then use a numerical function for thousands of volumes. Repeating symbolic substitution in a high-volume path adds overhead without changing the mathematics.

In [ ]:
p2_v=np.array(p2_volumes);p2_b_value=2.;p2_s_value=1.
p2_w_exact=p2_width_fn(p2_v,p2_b_value,p2_s_value)
p2_w_round=np.ceil(p2_w_exact*2)/2
p2_h_round=np.ceil((p2_v/p2_w_round**2)*2)/2
p2_cube=np.ceil(np.cbrt(p2_v)*2)/2
p2_rounded_cost=2*p2_b_value*p2_w_round**2+4*p2_s_value*p2_w_round*p2_h_round
p2_cube_cost=(2*p2_b_value+4*p2_s_value)*p2_cube**2
p2_exact_cost=2*p2_b_value*p2_w_exact**2+4*p2_s_value*p2_v/p2_w_exact
assert np.all(p2_w_round**2*p2_h_round>=p2_v-1e-9)
assert np.all(p2_rounded_cost>=p2_exact_cost-1e-9)
p2_report=pd.DataFrame({'volume_cm3':p2_v,'width_cm':p2_w_round,'height_cm':p2_h_round,'rounded_cost':p2_rounded_cost,'cube_cost':p2_cube_cost,'continuous_lower_bound':p2_exact_cost})
p2_test_v=np.linspace(100,30000,100)
p2_t=time.perf_counter();p2_symbolic=np.array([float(p2_optimal.subs({p2_V:float(v),p2_b:2,p2_s:1})) for v in p2_test_v]);p2_symbolic_time=time.perf_counter()-p2_t
p2_t=time.perf_counter();p2_numeric=p2_width_fn(p2_test_v,2,1);p2_numeric_time=time.perf_counter()-p2_t
assert np.allclose(p2_symbolic,p2_numeric)
display(p2_report);print({'100_substitutions_seconds':p2_symbolic_time,'vectorized_seconds':p2_numeric_time})
p2_report.to_csv(p2_out/'sympy_packaging_plan.csv',index=False)

### Explain the approximation honestly

Symbolic `==` compares expression structure, so the curvature check simplifies the difference to zero rather than assuming equivalent forms have identical trees. The rounded continuous optimum is feasible, but it need not be the exact optimum on the half-centimeter manufacturing grid. A small discrete neighborhood search can improve it. The comparison with a cube answers a cost question under unequal material prices, not a strength or environmental claim.

**Try:** set base and side prices equal. **Worked reasoning:** the continuous optimum becomes a cube because w³=V. If the implementation does not recover that special case, inspect the surface-area coefficients and volume substitution. Do not add a tolerance that hides an algebraic mistake.